In [ ]:
#Q1)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df=pd.read_csv("natural_gas_data/natural-gas_zip/archive/daily.csv")
print("Data:")
print(df)
print("Length of Data:",len(df))

df=df.dropna()
print("Length of Data after dropping null:",len(df))

y=df["Price"].values
x=np.arange(1,len(y))
epochs = 1500
print("\nLength of y: ",len(y))

minm=y.min()
maxm=y.max()
print(f"Minimum val: {minm},Maximum value: {maxm}")

y=(y-minm)/(maxm-minm) #normalize the ip range

Sequence_length=10

X=[]
Y=[]
for i in range(0,5900):
    list1=[]
    for j in range(i,i+Sequence_length):
        list1.append(y[j])
    X.append(list1)
    Y.append(y[j+1])

X=np.array(X)
Y=np.array(Y)

x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.10,random_state=42,
                                               shuffle=False,stratify=None)

class NGTimeSeries(Dataset):
    def __init__(self,x,y):
        self.x=torch.tensor(x,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)
        self.len=x.shape[0]
    def __getitem__(self, idx):
        return self.x[idx],self.y[idx]
    def __len__(self):
        return self.len

dataset=NGTimeSeries(x_train,y_train)

train_loader=DataLoader(dataset,batch_size=256,shuffle=True)

class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn=nn.RNN(input_size=1,hidden_size=5,num_layers=1,batch_first=True)
        self.fc1=nn.Linear(in_features=5,out_features=1)
    def forward(self, x):
        output, _status = self.rnn(x)        
        output = output[:, -1, :]            
        output = self.fc1(torch.relu(output))
        return output

model = RNNModel()

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
epochs = 1500

for i in range(epochs):
    for j, data in enumerate(train_loader):
        y_pred = model(data[:][0].view(-1, Sequence_length, 1)).reshape(-1)
        loss = criterion(y_pred, data[:][1])
        loss.backward()
        optimizer.step()
    if i % 50 == 0:
        print(i, "th iteration : ", loss)

test_set = NGTimeSeries(x_test,y_test)
test_pred = model(test_set[:][0].view(-1,10,1)).view(-1)

'''
print("\nSample Predictions vs Actual:\n")
for i in range(5):
    print(f"Input (10 days): {test_set[i][0]}")
    print(f"Actual (11th day): {test_set[i][1].item()}")
    print(f"Predicted: {test_pred[i].item()}")
'''
plt.plot(test_pred.detach().numpy(),label='predicted')
plt.plot(test_set[:][1].view(-1),label='original')
plt.legend()
plt.show()

y = y * (maxm - minm) + minm
y_pred = test_pred.detach().numpy() * (maxm - minm) + minm
plt.plot(y)
plt.plot(range(len(y)-len(y_pred), len(y)), y_pred)
plt.show()

Data:
            Date  Price
0     1997-01-07   3.82
1     1997-01-08   3.80
2     1997-01-09   3.61
3     1997-01-10   3.92
4     1997-01-13   4.00
...          ...    ...
5948  2020-08-26   2.52
5949  2020-08-27   2.52
5950  2020-08-28   2.46
5951  2020-08-31   2.30
5952  2020-09-01   2.22

[5953 rows x 2 columns]
Length of Data: 5953
Length of Data after dropping null: 5952

Length of y:  5952
Minimum val: 1.05,Maximum value: 18.48
0 th iteration :  tensor(0.5069, grad_fn=<MseLossBackward0>)


In [ ]:
import glob
import os
import string
import unicodedata
import random
import math
import torch
import torch.nn as nn

DATA_PATH = "data/names"
HIDDEN_SIZE = 128
LEARNING_RATE = 0.005
N_ITERS = 100000
PRINT_EVERY = 5000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

all_letters = string.ascii_letters 
n_letters = len(all_letters)

def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

def load_data(path):
    category_lines = {}
    all_categories = []

    for filename in glob.glob(path + '/*.txt'):
        category = os.path.splitext(os.path.basename(filename))[0]
        all_categories.append(category)

        with open(filename, encoding='utf-8') as f:
            lines = f.read().strip().split('\n')
            lines = [unicodeToAscii(line) for line in lines]
            category_lines[category] = lines

    return category_lines, all_categories

category_lines, all_categories = load_data(DATA_PATH)
n_categories = len(all_categories)

print("Loaded languages:", n_categories)

def letterToIndex(letter):
    return all_letters.find(letter)

def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_letters, device=DEVICE)
    for i, letter in enumerate(line):
        idx = letterToIndex(letter)
        if idx != -1:
            tensor[i][0][idx] = 1
    return tensor

class LSTMModel(nn.Module):
    def __init__(self,input_size,hidden_size,output_size):
        super().__init__()
        self.hidden_size=hidden_size
        self.lstm=nn.LSTM(input_size,hidden_size)
        self.fc=nn.Linear(hidden_size,output_size)
        self.softmax=nn.LogSoftmax(dim=1)
    def forward(self,input,hidden):
        output,hidden=self.lstm(input,hidden)
        output=self.fc(output[-1])
        output=self.softmax(output)

        return output,hidden
    def initHidden(self):
        return (torch.zeros(1,1,self.hidden_size,device=DEVICE),
                torch.zeros(1,1,self.hidden_size,device=DEVICE))

def randomTrainingExample():
    category = random.choice(all_categories)
    line = random.choice(category_lines[category])

    category_tensor = torch.tensor(
        [all_categories.index(category)],
        dtype=torch.long,
        device=DEVICE
    )

    line_tensor = lineToTensor(line)
    return category, line, category_tensor, line_tensor

lstm = LSTMModel(n_letters, HIDDEN_SIZE, n_categories).to(DEVICE)
optimizer=torch.optim.SGD(lstm.parameters(), lr=LEARNING_RATE)
criterion = nn.NLLLoss()

def train(category_tensor, line_tensor):
    hidden = lstm.initHidden()
    optimizer.zero_grad()

    for i in range(line_tensor.size(0)):
        output, hidden = lstm(line_tensor[i].unsqueeze(0), hidden)

    loss = criterion(output, category_tensor)
    loss.backward()

    optimizer.step()

    return output, loss.item()

for i in range(1, N_ITERS + 1):
    category, line, category_tensor, line_tensor = randomTrainingExample()
    output, loss = train(category_tensor, line_tensor)

    if i % PRINT_EVERY == 0:
        guess_i = torch.argmax(output).item()
        guess = all_categories[guess_i]

        correct = " Correct " if guess == category else f"Wrong- Actual: ({category})"

        print(f"{i} |  Loss: {loss:.4f} | {line} → {guess} {correct}")

torch.save(lstm.state_dict(), "lstm_name_classifier.pth")

print("\nModel saved!")

def predict(input_line, n_predictions=3):
    with torch.no_grad():
        line_tensor = lineToTensor(input_line)
        hidden = lstm.initHidden()

        for i in range(line_tensor.size(0)):
            output, hidden = lstm(line_tensor[i].unsqueeze(0), hidden)

        topv, topi = output.topk(n_predictions, 1, True)

        predictions = []
        for i in range(n_predictions):
            value = topv[0][i].item()
            category_index = topi[0][i].item()
            predictions.append((all_categories[category_index], value))

        return predictions


print("\nPredictions:")
test_names = ["Schmidt", "Garcia", "Ivanov", "Kim"]
pred=["German","Spanish","Russian","Korean"]

for i in range(len(test_names)):
    name=test_names[i]
    preds = predict(name)
    print(f"\n{name}:")
    max_sc=float('-inf')
    max_lang=None
    for lang, score in preds:
        print(f"  {lang} ({score:.4f})")
        if (score>max_sc):
            max_sc=score
            max_lang=lang
    print("Predicted Language: ", max_lang)
    print("Actual Language: ", pred[i])

Loaded languages: 18
5000 |  Loss: 2.9050 | Chester → Polish Wrong- Actual: (English)
10000 |  Loss: 2.9256 | Durant → Russian Wrong- Actual: (French)
15000 |  Loss: 2.9512 | Colman → Greek Wrong- Actual: (Irish)
20000 |  Loss: 2.8135 | Nishiwaki → Russian Wrong- Actual: (Japanese)
25000 |  Loss: 2.9797 | Mikhail → Japanese Wrong- Actual: (Arabic)
30000 |  Loss: 1.6761 | Thao → Vietnamese  Correct 
35000 |  Loss: 1.3960 | Rudawski → Polish  Correct 
40000 |  Loss: 2.7448 | OToole → English Wrong- Actual: (Irish)
45000 |  Loss: 1.9671 | Brown → Irish Wrong- Actual: (Scottish)
50000 |  Loss: 1.7220 | Rios → Greek Wrong- Actual: (Portuguese)
55000 |  Loss: 1.3808 | Garcia → Portuguese  Correct 
60000 |  Loss: 2.0305 | Cennetig → Scottish Wrong- Actual: (Irish)
65000 |  Loss: 0.8336 | Niall → Irish  Correct 
70000 |  Loss: 0.7876 | Abarca → Spanish  Correct 
75000 |  Loss: 6.3759 | Sarkozi → Polish Wrong- Actual: (French)
80000 |  Loss: 0.0936 | Zdunowski → Polish  Correct 
85000 |  Loss: 

In [ ]:
#Q3
import torch
import torch.nn as nn
import random
import string

all_characters = string.printable
n_chars = len(all_charS","Russiacters)

HIDDEN_SIZE = 128
LEARNING_RATE = 0.005
N_ITERS = 20000
PRINT_EVERY = 2000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

text = """hello world
deep learning"""

def charToIndex(c):
    return all_characters.find(c)

def charToTensor(c):
    tensor = torch.zeros(1, n_chars, device=DEVICE)
    tensor[0][charToIndex(c)] = 1
    return tensor

def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_chars, device=DEVICE)
    for i, c in enumerate(line):
        tensor[i][0][charToIndex(c)] = 1
    return tensor

class LSTMModel(nn.Module):
    def __init__(self,input_size,hidden_size,output_size):
        super().__init__()
        self.hidden_size=hidden_size
        self.lstm=nn.LSTM(input_size,hidden_size)
        self.fc=nn.Linear(hidden_size,output_size)
    def forward(self,input,hidden):
        output,hidden=self.lstm(input,hidden)
        output=self.fc(output[-1])
        return output,hidden
    def initHidden(self):
        return (torch.zeros(1,1,self.hidden_size,device=DEVICE),
                torch.zeros(1,1,self.hidden_size,device=DEVICE))

def randomTrainingExample():
    start_index = random.randint(0, len(text) - 2)
    end_index = start_index + random.randint(5, 15)

    chunk = text[start_index:end_index]

    input_seq = chunk[:-1]
    target_seq = chunk[1:]

    input_tensor = lineToTensor(input_seq)
    target_tensor = torch.tensor(
        [charToIndex(c) for c in target_seq],
        dtype=torch.long,
        device=DEVICE
    )

    return input_tensor, target_tensor

lstm = LSTMModel(n_chars, HIDDEN_SIZE, n_chars).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm.parameters(), lr=LEARNING_RATE)

def train(input_tensor, target_tensor):
    hidden = lstm.initHidden()
    optimizer.zero_grad()

    loss = 0

    for i in range(input_tensor.size(0)):
        output, hidden = lstm(input_tensor[i].unsqueeze(0), hidden)
        loss += criterion(output, target_tensor[i].unsqueeze(0))

    loss.backward()

    optimizer.step()

    return loss.item() / input_tensor.size(0)

for i in range(1, N_ITERS + 1):
    input_tensor, target_tensor = randomTrainingExample()
    loss = train(input_tensor, target_tensor)

    if i % PRINT_EVERY == 0:
        print(f"Iter {i} | Loss: {loss:.4f}")

def generate(start_str="he", predict_len=100):
    with torch.no_grad():
        input_tensor = lineToTensor(start_str)
        hidden = lstm.initHidden()

        # feed initial string
        for i in range(len(start_str) - 1):
            _, hidden = lstm(input_tensor[i].unsqueeze(0), hidden)

        last_char = input_tensor[-1]
        output_str = start_str

        for _ in range(predict_len):
            output, hidden = lstm(last_char.unsqueeze(0), hidden)

            probs = torch.softmax(output, dim=1)
            topi = torch.multinomial(probs, 1)[0]

            predicted_char = all_characters[topi]
            output_str += predicted_char

            last_char = charToTensor(predicted_char)

        return output_str

print("\nGenerated Text:\n")
print(generate("deep ", 8))
print(generate("hello ", 5))
print(generate("world ", 5))

Iter 2000 | Loss: 0.1375
Iter 4000 | Loss: 0.1146
Iter 6000 | Loss: 0.0953
Iter 8000 | Loss: 0.0001
Iter 10000 | Loss: 0.3875
Iter 12000 | Loss: 0.2353
Iter 14000 | Loss: 0.2188
Iter 16000 | Loss: 0.0000
Iter 18000 | Loss: 0.0000
Iter 20000 | Loss: 0.0002

Generated Text:

deep learning
hello world
world learn
